# Notebook 6.1: Time Series Models with Comprehensive Metrics

## Objetivo

Este notebook implementa modelos de **series temporales** (LSTM, ARIMA, SARIMA) para las **72 configuraciones** de series temporales (filas 37-108) del marco teórico.

**Modelos implementados**:
- **LSTM**: Long Short-Term Memory (Redes Neuronales Recurrentes)
- **ARIMA**: AutoRegressive Integrated Moving Average  
- **SARIMA**: Seasonal ARIMA

**Total combinaciones**: 72 configuraciones × 3 modelos = **216 combinaciones**

**Metodología**: 
- Cross-validation para series temporales (TimeSeriesSplit)
- Transformaciones específicas para cada modelo
- Métricas comprensivas (RMSE, MAE, NSE, KGE, PBIAS_%, MAPE_%)

**Autor**: Víctor Galán  
**Fecha**: 2026-02-14

## 1. Imports y Configuración

In [1]:
import pandas as pd
import numpy as np
import os
import json
import warnings
from tqdm import tqdm
from datetime import datetime

# Machine Learning
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Deep Learning (LSTM)
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers, Sequential
    from tensorflow.keras.callbacks import EarlyStopping
    TENSORFLOW_AVAILABLE = True
except ImportError:
    TENSORFLOW_AVAILABLE = False
    print("⚠️  TensorFlow not available. Install with: pip install tensorflow")

# Time Series (ARIMA/SARIMA)
try:
    from statsmodels.tsa.arima.model import ARIMA
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    from statsmodels.tsa.stattools import adfuller
    from statsmodels.tools.sm_exceptions import ConvergenceWarning
    STATSMODELS_AVAILABLE = True
except ImportError:
    STATSMODELS_AVAILABLE = False
    print("⚠️  Statsmodels not available. Install with: pip install statsmodels")

# Scipy for statistical calculations
from scipy import stats

warnings.filterwarnings('ignore')
if STATSMODELS_AVAILABLE:
    warnings.filterwarnings('ignore', category=ConvergenceWarning)

print("=" * 80)
print("NOTEBOOK 6.1: TIME SERIES MODELS WITH COMPREHENSIVE METRICS")
print("=" * 80)
print(f"✅ Imports loaded successfully")
print(f"   numpy version: {np.__version__}")
print(f"   pandas version: {pd.__version__}")

print("\n📊 Time Series Models Availability:")
print(f"   TensorFlow (LSTM): {'✅' if TENSORFLOW_AVAILABLE else '❌'}")
print(f"   Statsmodels (ARIMA/SARIMA): {'✅' if STATSMODELS_AVAILABLE else '❌'}")

if TENSORFLOW_AVAILABLE:
    print(f"\n🎯 TensorFlow version: {tf.__version__}")

NOTEBOOK 6.1: TIME SERIES MODELS WITH COMPREHENSIVE METRICS
✅ Imports loaded successfully
   numpy version: 2.2.5
   pandas version: 2.3.3

📊 Time Series Models Availability:
   TensorFlow (LSTM): ✅
   Statsmodels (ARIMA/SARIMA): ✅

🎯 TensorFlow version: 2.20.0


## 2. Funciones de Métricas Hidrológicas

Implementamos las mismas métricas del notebook 6 más métricas adicionales relevantes para modelos de series temporales.

In [2]:
def calculate_nse(y_true, y_pred):
    """
    Calculate Nash-Sutcliffe Efficiency (NSE).
    
    NSE = 1 - [Σ(y_true - y_pred)²] / [Σ(y_true - mean(y_true))²]
    
    Range: -∞ to 1 (perfect fit)
    """
    numerator = np.sum((y_true - y_pred) ** 2)
    denominator = np.sum((y_true - np.mean(y_true)) ** 2)
    
    if denominator == 0:
        return np.nan
    
    nse = 1 - (numerator / denominator)
    return nse

def calculate_kge(y_true, y_pred):
    """
    Calculate Kling-Gupta Efficiency (KGE).
    
    KGE = 1 - sqrt[(r-1)² + (α-1)² + (β-1)²]
    
    Where:
    - r: correlation coefficient
    - α: ratio of standard deviations
    - β: ratio of means
    """
    r = np.corrcoef(y_true, y_pred)[0, 1]
    alpha = np.std(y_pred) / np.std(y_true) if np.std(y_true) > 0 else np.nan
    beta = np.mean(y_pred) / np.mean(y_true) if np.mean(y_true) != 0 else np.nan
    
    kge = 1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2)
    return kge

def calculate_pbias(y_true, y_pred):
    """
    Calculate Percent Bias (PBIAS).
    
    PBIAS = [Σ(y_true - y_pred) / Σ(y_true)] × 100
    """
    if np.sum(y_true) == 0:
        return np.nan
    
    pbias = (np.sum(y_true - y_pred) / np.sum(y_true)) * 100
    return pbias

def calculate_mape(y_true, y_pred):
    """
    Calculate Mean Absolute Percentage Error (MAPE).
    """
    mask = y_true != 0
    if not np.any(mask):
        return np.nan
    
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    return mape

def calculate_all_metrics(y_true, y_pred):
    """
    Calculate all metrics at once.
    """
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    
    nse = calculate_nse(y_true, y_pred)
    kge = calculate_kge(y_true, y_pred)
    pbias = calculate_pbias(y_true, y_pred)
    mape = calculate_mape(y_true, y_pred)
    
    return {
        'RMSE': rmse,
        'MAE': mae,
        'NSE': nse,
        'KGE': kge,
        'PBIAS_%': pbias,
        'MAPE_%': mape
    }

print("✅ Metric functions defined:")
print("   - NSE (Nash-Sutcliffe Efficiency)")
print("   - KGE (Kling-Gupta Efficiency)")  
print("   - PBIAS (Percent Bias)")
print("   - MAPE (Mean Absolute Percentage Error)")
print("   - RMSE, MAE (standard metrics)")

✅ Metric functions defined:
   - NSE (Nash-Sutcliffe Efficiency)
   - KGE (Kling-Gupta Efficiency)
   - PBIAS (Percent Bias)
   - MAPE (Mean Absolute Percentage Error)
   - RMSE, MAE (standard metrics)


## 3. Funciones de Transformación de Datos

In [3]:
def create_lstm_sequences(X, y, n_steps=7):
    """
    Create sequences for LSTM model.
    
    Parameters:
    -----------
    X : array-like
        Feature matrix
    y : array-like
        Target variable
    n_steps : int
        Number of previous timesteps to use
        
    Returns:
    --------
    X_seq, y_seq : arrays
        Sequences for LSTM training
    """
    X_seq, y_seq = [], []
    
    for i in range(n_steps, len(X)):
        X_seq.append(X[i-n_steps:i])
        y_seq.append(y[i])
        
    return np.array(X_seq), np.array(y_seq)

def prepare_arima_data(df, feature_cols, target_col):
    """
    Prepare data for ARIMA/SARIMA by aggregating to daily level.
    
    Parameters:
    -----------
    df : DataFrame
        Time series data with Date column
    feature_cols : list
        List of feature column names
    target_col : str
        Target variable column name
        
    Returns:
    --------
    df_daily : DataFrame
        Daily aggregated data
    """
    if 'Date' not in df.columns:
        raise ValueError("DataFrame must have 'Date' column")
    
    # Ensure Date is datetime
    if not pd.api.types.is_datetime64_any_dtype(df['Date']):
        df['Date'] = pd.to_datetime(df['Date'])
    
    # Aggregate to daily (take mean of AM/PM readings)
    agg_dict = {target_col: 'mean'}
    for feat in feature_cols:
        if feat in df.columns:
            agg_dict[feat] = 'mean'
    
    df_daily = df.groupby(df['Date'].dt.date).agg(agg_dict).reset_index()
    df_daily.columns = ['Date'] + list(df_daily.columns[1:])
    df_daily['Date'] = pd.to_datetime(df_daily['Date'])
    
    return df_daily

print("✅ Data transformation functions defined:")
print("   - create_lstm_sequences(): Create sequences for LSTM")
print("   - prepare_arima_data(): Aggregate to daily for ARIMA/SARIMA")

✅ Data transformation functions defined:
   - create_lstm_sequences(): Create sequences for LSTM
   - prepare_arima_data(): Aggregate to daily for ARIMA/SARIMA


## 4. Carga de Datos

In [4]:
# Load merged data
base_path = "../../data/clean/merged_data/"

if not os.path.exists(base_path):
    raise FileNotFoundError(f"Path {base_path} does not exist. Run 0-ebro-data-cleaning.ipynb first.")

join_dict = {}
for filename in os.listdir(base_path):
    # if filename.endswith(".json"): # read data  WITH outliers
    if filename.endswith("-without-outliers.json"): # read data WITH PREFIX: -without-outliers
        basin_name = filename.replace(".json", "")
        df = pd.read_json(os.path.join(base_path, filename), orient="records")
        
        # Ensure Date column exists and is datetime
        if 'Date' in df.columns:
            df['Date'] = pd.to_datetime(df['Date'])
            df = df.sort_values('Date').reset_index(drop=True)
        
        join_dict[basin_name] = df

print("=" * 80)
print("DATA LOADING")
print("=" * 80)
print(f"✅ Loaded {len(join_dict)} basins")
print(f"📊 Basins: {sorted(list(join_dict.keys()))}")

# Sample basin for verification
sample_basin = list(join_dict.keys())[0]
print(f"\n📋 Sample basin: {sample_basin}")
print(f"   Shape: {join_dict[sample_basin].shape}")
print(f"   Columns: {len(join_dict[sample_basin].columns)}")

if 'Date' in join_dict[sample_basin].columns:
    date_range = f"{join_dict[sample_basin]['Date'].min()} to {join_dict[sample_basin]['Date'].max()}"
    print(f"   Date range: {date_range}")

DATA LOADING
✅ Loaded 18 basins
📊 Basins: ['Anduña-without-outliers', 'Bailin-without-outliers', 'Deza-without-outliers', 'Flamisell-without-outliers', 'Garona-without-outliers', 'Isuela-without-outliers', 'Larraun-without-outliers', 'Nela-without-outliers', 'Oroncillo-without-outliers', 'Rudron-without-outliers', 'Sangüesa-without-outliers', 'Subialde-without-outliers', 'Tiron-without-outliers', 'Ubagua-without-outliers', 'Urrobi-without-outliers', 'Vallfarrera-without-outliers', 'Yanguas-without-outliers', 'Zatoya-without-outliers']

📋 Sample basin: Ubagua-without-outliers
   Shape: (2754, 73)
   Columns: 73
   Date range: 2020-10-01 00:00:00 to 2025-09-30 00:00:00


## 5. Crear Variables Sintéticas

In [5]:
# Create synthetic variables for all basins
synthetic_vars_created = {
    'Dif_t': 0,
    'Dif_HR': 0,
    'Dif_q': 0,
    'Dif_y': 0,
    'Dif_qr': 0,
    'Dif_qs': 0
}

for basin_name, df in join_dict.items():
    # Temperature difference
    if 'tmax' in df.columns and 'tmin' in df.columns:
        df['Dif_t'] = df['tmax'] - df['tmin']
        synthetic_vars_created['Dif_t'] += 1

    # Humidity difference
    if 'Hrmax' in df.columns and 'Hrmin' in df.columns:
        df['Dif_HR'] = df['Hrmax'] - df['Hrmin']
        synthetic_vars_created['Dif_HR'] += 1

    # Flow difference
    if 'qmax' in df.columns and 'qmin' in df.columns:
        df['Dif_q'] = df['qmax'] - df['qmin']
        synthetic_vars_created['Dif_q'] += 1

    # Water level difference
    if 'ymax' in df.columns and 'ymin' in df.columns:
        df['Dif_y'] = df['ymax'] - df['ymin']
        synthetic_vars_created['Dif_y'] += 1
    
    # Soil moisture differences (SMAP)
    if 'sm_rootzone_max' in df.columns and 'sm_rootzone_min' in df.columns:
        df['Dif_qr'] = df['sm_rootzone_max'] - df['sm_rootzone_min']
        synthetic_vars_created['Dif_qr'] += 1
    
    if 'sm_surface_max' in df.columns and 'sm_surface_min' in df.columns:
        df['Dif_qs'] = df['sm_surface_max'] - df['sm_surface_min']
        synthetic_vars_created['Dif_qs'] += 1

    # Update dictionary
    join_dict[basin_name] = df

print("✅ Synthetic variables created:")
for var, count in synthetic_vars_created.items():
    print(f"   - {var}: Created in {count}/{len(join_dict)} basins")

✅ Synthetic variables created:
   - Dif_t: Created in 18/18 basins
   - Dif_HR: Created in 15/18 basins
   - Dif_q: Created in 18/18 basins
   - Dif_y: Created in 18/18 basins
   - Dif_qr: Created in 18/18 basins
   - Dif_qs: Created in 18/18 basins


## 6. Cargar Asignaciones de Clusters

In [6]:
# Cluster assignments from notebook 2 (K-Means clustering, k=3)

# CAUTION!!!! THESE ARE THE CLUSTERS BEFORE REMOVING OUTLIERS
# cluster_assignments = {
#     'Cluster_0': ['Deza', 'Flamisell', 'Zidacos', 'Subialde', 'Zatoya', 'Vallfarrera', 
#                   'Isuela', 'Urrobi', 'Bailin', 'Oroncillo', 'Sangüesa'],
#     'Cluster_1': ['Larraun', 'Trueba', 'Nela'],
#     'Cluster_2': ['Yanguas', 'Anduña', 'Tiron', 'Rudron', 'Garona', 'Ubagua']
# }

# CAUTION!!!! THESE ARE THE CLUSTERS AFTER REMOVING OUTLIERS
cluster_assignments = {
    'Cluster 0': ['Ubagua-without-outliers', 'Subialde-without-outliers', 'Oroncillo-without-outliers', 
                  'Anduña-without-outliers', 'Isuela-without-outliers', 'Yanguas-without-outliers', 
                  'Deza-without-outliers', 'Zatoya-without-outliers', 'Flamisell-without-outliers', 
                  'Vallfarrera-without-outliers', 'Bailin-without-outliers', 'Sangüesa-without-outliers', 
                  'Urrobi-without-outliers'],
    'Cluster 1': ['Tiron-without-outliers', 'Garona-without-outliers', 'Rudron-without-outliers'],
    'Cluster 2': ['Nela-without-outliers', 'Larraun-without-outliers']
}


# Create reverse mapping: basin -> cluster
basin_to_cluster = {}
for cluster_name, basins in cluster_assignments.items():
    for basin in basins:
        basin_to_cluster[basin] = cluster_name

print("=" * 80)
print("CLUSTER ASSIGNMENTS")
print("=" * 80)
print("Cluster assignments from notebook 2 (K-Means, k=3):")
for cluster, basins in cluster_assignments.items():
    print(f"  {cluster}: {len(basins)} basins - {basins}")

# Create cluster-specific datasets
cluster_data = {}
for cluster_name, basins in cluster_assignments.items():
    cluster_dfs = [join_dict[basin].copy() for basin in basins if basin in join_dict]
    if cluster_dfs:
        cluster_df = pd.concat(cluster_dfs, ignore_index=True)
        cluster_data[cluster_name] = cluster_df
        print(f"\n{cluster_name}: {len(cluster_df):,} observations from {len(basins)} basins")

print(f"\n✅ Cluster data prepared for modeling")

CLUSTER ASSIGNMENTS
Cluster assignments from notebook 2 (K-Means, k=3):
  Cluster 0: 13 basins - ['Ubagua-without-outliers', 'Subialde-without-outliers', 'Oroncillo-without-outliers', 'Anduña-without-outliers', 'Isuela-without-outliers', 'Yanguas-without-outliers', 'Deza-without-outliers', 'Zatoya-without-outliers', 'Flamisell-without-outliers', 'Vallfarrera-without-outliers', 'Bailin-without-outliers', 'Sangüesa-without-outliers', 'Urrobi-without-outliers']
  Cluster 1: 3 basins - ['Tiron-without-outliers', 'Garona-without-outliers', 'Rudron-without-outliers']
  Cluster 2: 2 basins - ['Nela-without-outliers', 'Larraun-without-outliers']

Cluster 0: 37,344 observations from 13 basins

Cluster 1: 8,507 observations from 3 basins

Cluster 2: 5,715 observations from 2 basins

✅ Cluster data prepared for modeling


## 7. Mapeo de Variables

In [7]:
# Variable mapping: Paper variables -> Real column names
var_mapping = {
    # Climate/temporal variables
    'P': 'P',
    'tav': 'tav',
    'tmax': 'tmax',
    'tmin': 'tmin',
    'Hrav': 'Hrav',
    'Hrmax': 'Hrmax',
    'Hrmin': 'Hrmin',
    'wv': 'wv',
    'wvmax': 'wvmax',
    'SolarRad': 'SolarRad',
    'et0': 'et0',
    
    # SMAP soil moisture
    'Av_qr': 'sm_rootzone_av',
    'Av_qs': 'sm_surface_av',
    'Dif_qr': 'Dif_qr',
    'Dif_qs': 'Dif_qs',
    
    # Synthetic variables
    'Dif_t': 'Dif_t',
    'Dif_HR': 'Dif_HR',
    'Dif_q': 'Dif_q',
    'Dif_y': 'Dif_y',
    
    # Target variables
    'ymin': 'ymin',
    'yav': 'yav',
    'ymax': 'ymax',
    'qmin': 'qmin',
    'qav': 'qav',
    'qmax': 'qmax'
}

def map_variables(var_list):
    """Map paper variable names to real column names"""
    return [var_mapping.get(v, v) for v in var_list]

print(f"✅ Variable mapping defined: {len(var_mapping)} variables")

✅ Variable mapping defined: 25 variables


## 8. Especificación de Configuraciones de Series Temporales (Filas 37-108)

Definimos las 72 configuraciones de series temporales que se ejecutarán con LSTM, ARIMA y SARIMA.

In [8]:
# Define time series model specifications (rows 37-108)
model_specifications = []

# Targets para series temporales
ts_targets = ['ymin', 'yav', 'ymax', 'Dif_y', 'qmin', 'qav', 'qmax', 'Dif_q']

# ============================================================================
# TODAS LAS CUENCAS - Variables completas (filas 37-44)
# ============================================================================
ts_completo_vars = ['P', 'tmax', 'tav', 'tmin', 'Hrav', 'Hrmax', 'Hrmin',
                    'wv', 'wvmax', 'SolarRad', 'et0', 'Av_qr', 'Av_qs']

for i, target in enumerate(ts_targets, 37):
    model_specifications.append({
        'Num': i,
        'Aproximación': 'Series temporales',
        'Enfoque': 'Todas las cuencas',
        'Modelo': 'Variables primarias',
        'Valores_a_usar': 'N/A',
        'Target': target,
        'Features': ts_completo_vars,
        'Dataset': 'global'
    })

# ============================================================================
# TODAS LAS CUENCAS - Reducido (filas 45-52)
# ============================================================================
ts_reducido_vars = ['P', 'et0', 'Av_qr', 'Av_qs']

for i, target in enumerate(ts_targets, 45):
    model_specifications.append({
        'Num': i,
        'Aproximación': 'Series temporales',
        'Enfoque': 'Todas las cuencas',
        'Modelo': 'Variables primarias',
        'Valores_a_usar': 'N/A',
        'Target': target,
        'Features': ts_reducido_vars,
        'Dataset': 'global'
    })

# ============================================================================
# TODAS LAS CUENCAS - Mínimo (filas 53-60)
# ============================================================================
ts_minimo_vars = ['P', 'Av_qr', 'Av_qs']

for i, target in enumerate(ts_targets, 53):
    model_specifications.append({
        'Num': i,
        'Aproximación': 'Series temporales',
        'Enfoque': 'Todas las cuencas',
        'Modelo': 'Variables primarias',
        'Valores_a_usar': 'N/A',
        'Target': target,
        'Features': ts_minimo_vars,
        'Dataset': 'global'
    })

# ============================================================================
# POR CLUSTER GA - Completo (filas 61-68)
# ============================================================================
for i, target in enumerate(ts_targets, 61):
    model_specifications.append({
        'Num': i,
        'Aproximación': 'Series temporales',
        'Enfoque': 'Para cada cluster según GA',
        'Modelo': 'Variables primarias',
        'Valores_a_usar': 'N/A',
        'Target': target,
        'Features': ts_completo_vars,
        'Dataset': 'clusters'
    })

# ============================================================================
# POR CLUSTER GA - Reducido (filas 69-76)
# ============================================================================
for i, target in enumerate(ts_targets, 69):
    model_specifications.append({
        'Num': i,
        'Aproximación': 'Series temporales',
        'Enfoque': 'Para cada cluster según GA',
        'Modelo': 'Variables primarias',
        'Valores_a_usar': 'N/A',
        'Target': target,
        'Features': ts_reducido_vars,
        'Dataset': 'clusters'
    })

# ============================================================================
# POR CLUSTER GA - Mínimo (filas 77-84)
# ============================================================================
for i, target in enumerate(ts_targets, 77):
    model_specifications.append({
        'Num': i,
        'Aproximación': 'Series temporales',
        'Enfoque': 'Para cada cluster según GA',
        'Modelo': 'Variables primarias',
        'Valores_a_usar': 'N/A',
        'Target': target,
        'Features': ts_minimo_vars,
        'Dataset': 'clusters'
    })

# ============================================================================
# POR CLUSTER CN - Completo (filas 85-92)
# ============================================================================
for i, target in enumerate(ts_targets, 85):
    model_specifications.append({
        'Num': i,
        'Aproximación': 'Series temporales',
        'Enfoque': 'Para cada cluster según CN',
        'Modelo': 'Variables primarias',
        'Valores_a_usar': 'N/A',
        'Target': target,
        'Features': ts_completo_vars,
        'Dataset': 'clusters'
    })

# ============================================================================
# POR CLUSTER CN - Reducido (filas 93-100)
# ============================================================================
for i, target in enumerate(ts_targets, 93):
    model_specifications.append({
        'Num': i,
        'Aproximación': 'Series temporales',
        'Enfoque': 'Para cada cluster según CN',
        'Modelo': 'Variables primarias',
        'Valores_a_usar': 'N/A',
        'Target': target,
        'Features': ts_reducido_vars,
        'Dataset': 'clusters'
    })

# ============================================================================
# POR CLUSTER CN - Mínimo (filas 101-108)
# ============================================================================
for i, target in enumerate(ts_targets, 101):
    model_specifications.append({
        'Num': i,
        'Aproximación': 'Series temporales',
        'Enfoque': 'Para cada cluster según CN',
        'Modelo': 'Variables primarias',
        'Valores_a_usar': 'N/A',
        'Target': target,
        'Features': ts_minimo_vars,
        'Dataset': 'clusters'
    })

print(f"✅ Series temporales: {len(model_specifications)} configuraciones (filas 37-108)")
print(f"   - Todas las cuencas (completo, reducido, mínimo): 24 configuraciones")
print(f"   - Por cluster GA (3 variantes × 8 targets): 24 configuraciones")
print(f"   - Por cluster CN (3 variantes × 8 targets): 24 configuraciones")
print(f"\n🎯 Total: {len(model_specifications)} configuraciones × 3 modelos (LSTM, ARIMA, SARIMA) = {len(model_specifications) * 3} combinaciones")

✅ Series temporales: 72 configuraciones (filas 37-108)
   - Todas las cuencas (completo, reducido, mínimo): 24 configuraciones
   - Por cluster GA (3 variantes × 8 targets): 24 configuraciones
   - Por cluster CN (3 variantes × 8 targets): 24 configuraciones

🎯 Total: 72 configuraciones × 3 modelos (LSTM, ARIMA, SARIMA) = 216 combinaciones


## 9. Funciones de Entrenamiento LSTM

In [9]:
def train_lstm_model(df, feature_cols, target_col, n_steps=7, epochs=50, verbose=0):
    """
    Train LSTM model for time series prediction.

    Parameters
    ----------
    df : DataFrame
        Time series data (must be sorted by Date)
    feature_cols : list
        List of feature column names
    target_col : str
        Target variable column name
    n_steps : int
        Number of previous timesteps to use
    epochs : int
        Number of training epochs
    verbose : int
        Verbosity level (0 = silent)

    Returns
    -------
    metrics : dict
        Dictionary with all metrics
    model : keras.Model
        Trained LSTM model
    """

    if not TENSORFLOW_AVAILABLE:
        raise ImportError("TensorFlow not available. Install with: pip install tensorflow")

    # Check for missing columns
    missing_features = [col for col in feature_cols if col not in df.columns]
    if missing_features:
        raise ValueError(f"Missing features: {missing_features}")

    if target_col not in df.columns:
        raise ValueError(f"Target '{target_col}' not in DataFrame")

    # Drop NaN
    cols_to_check = feature_cols + [target_col]
    df_clean = df[cols_to_check].dropna()

    if len(df_clean) < n_steps + 100:
        raise ValueError(f"Insufficient data: {len(df_clean)} samples (need at least {n_steps + 100})")

    # Scale features and target
    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()

    X_scaled = scaler_X.fit_transform(df_clean[feature_cols])
    y_scaled = scaler_y.fit_transform(df_clean[[target_col]])

    # Create sequences
    X_seq, y_seq = create_lstm_sequences(X_scaled, y_scaled.flatten(), n_steps=n_steps)

    # Time series split (80/20)
    split_idx = int(0.8 * len(X_seq))
    X_train, X_test = X_seq[:split_idx], X_seq[split_idx:]
    y_train, y_test = y_seq[:split_idx], y_seq[split_idx:]

    # Build LSTM model
    model = Sequential([
        layers.LSTM(64, activation='relu', return_sequences=True,
                    input_shape=(n_steps, len(feature_cols))),
        layers.Dropout(0.2),
        layers.LSTM(32, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(16, activation='relu'),
        layers.Dense(1)
    ])

    model.compile(optimizer='adam', loss='mse', metrics=['mae'])

    # Train with early stopping
    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=0
    )

    model.fit(
        X_train, y_train,
        epochs=epochs,
        batch_size=32,
        validation_split=0.1,
        callbacks=[early_stop],
        verbose=verbose
    )

    # Predictions
    y_test_pred = model.predict(X_test, verbose=0)

    # Inverse transform
    y_test_actual = scaler_y.inverse_transform(y_test.reshape(-1, 1)).flatten()
    y_test_pred_actual = scaler_y.inverse_transform(y_test_pred).flatten()

    # Calculate metrics
    metrics = calculate_all_metrics(y_test_actual, y_test_pred_actual)

    return metrics, model


print("✅ LSTM training function defined")


✅ LSTM training function defined


## 10. Funciones de Entrenamiento ARIMA/SARIMA

In [10]:
def train_arima_model(df, feature_cols, target_col, order=(1, 1, 1), verbose=0):
    """
    Train SARIMAX model with exogenous variables.

    Parameters
    ----------
    df : DataFrame
        Time series data with Date column
    feature_cols : list
        List of feature column names (exogenous variables)
    target_col : str
        Target variable column name
    order : tuple
        ARIMA order (p, d, q)
    verbose : int
        Verbosity level

    Returns
    -------
    metrics : dict
        Dictionary with all metrics
    model : SARIMAXResults
        Fitted SARIMAX model
    """

    if not STATSMODELS_AVAILABLE:
        raise ImportError("Statsmodels not available. Install with: pip install statsmodels")

    # Prepare daily aggregated data
    df_daily = prepare_arima_data(df, feature_cols, target_col)

    # Check for missing columns
    missing_features = [col for col in feature_cols if col not in df_daily.columns]
    if missing_features:
        raise ValueError(f"Missing features: {missing_features}")

    # Drop NaN
    cols_to_check = feature_cols + [target_col]
    df_clean = df_daily[cols_to_check].dropna()

    if len(df_clean) < 100:
        raise ValueError(f"Insufficient daily data: {len(df_clean)} days (need at least 100)")

    # Time series split (80/20)
    split_idx = int(0.8 * len(df_clean))
    train_data = df_clean.iloc[:split_idx]
    test_data = df_clean.iloc[split_idx:]

    # Check stationarity (for d parameter)
    try:
        adf_result = adfuller(train_data[target_col].dropna())
        is_stationary = adf_result[1] <= 0.05

        # Adjust d if non-stationary
        if not is_stationary and order[1] == 0:
            order = (order[0], 1, order[2])
    except:
        pass  # Use default order if test fails

    # Fit SARIMAX model
    try:
        model = SARIMAX(
            train_data[target_col],
            exog=train_data[feature_cols],
            order=order,
            enforce_stationarity=False,
            enforce_invertibility=False
        )

        fitted_model = model.fit(disp=False, maxiter=200)

        # Forecast on test set
        forecast = fitted_model.forecast(
            steps=len(test_data),
            exog=test_data[feature_cols]
        )

        # Calculate metrics
        y_test_actual = test_data[target_col].values
        y_test_pred = forecast.values

        metrics = calculate_all_metrics(y_test_actual, y_test_pred)

        return metrics, fitted_model

    except Exception as e:
        # If SARIMAX fails, return NaN metrics
        return {
            'RMSE': np.nan,
            'MAE': np.nan,
            'NSE': np.nan,
            'KGE': np.nan,
            'PBIAS_%': np.nan,
            'MAPE_%': np.nan,
            'Error': str(e)
        }, None


print("✅ ARIMA/SARIMA training function defined")


✅ ARIMA/SARIMA training function defined


## 11. Preparar Dataset Global

Combinamos todas las cuencas en un dataset global para las configuraciones "Todas las cuencas".

In [11]:
# Combine all basins into global dataset
all_basins_data = []

for basin_name, df in join_dict.items():
    df_copy = df.copy()
    df_copy['Basin'] = basin_name
    all_basins_data.append(df_copy)

global_df = pd.concat(all_basins_data, ignore_index=True)

# Sort by date
if 'Date' in global_df.columns:
    global_df = global_df.sort_values('Date').reset_index(drop=True)

print("=" * 80)
print("GLOBAL DATASET PREPARATION")
print("=" * 80)
print("✅ Global dataset created")
print(f"   Total observations: {len(global_df):,}")
print(f"   Basins: {len(join_dict)}")
print(f"   Columns: {len(global_df.columns)}")

if 'Date' in global_df.columns:
    print(f"   Date range: {global_df['Date'].min()} to {global_df['Date'].max()}")


GLOBAL DATASET PREPARATION
✅ Global dataset created
   Total observations: 51,566
   Basins: 18
   Columns: 80
   Date range: 2020-10-01 00:00:00 to 2025-09-30 00:00:00


## 12. Ejecución de Modelos

**ADVERTENCIA**: Esta celda entrena **216 modelos** (72 configuraciones × 3 modelos).  
El tiempo estimado de ejecución es **2-4 horas** dependiendo del hardware.

Se utilizará `tqdm` para mostrar el progreso.

In [12]:
# Initialize results storage
all_results = []

# ML models to execute
ml_models = ['LSTM', 'ARIMA', 'SARIMA']

# Total combinations
total_combinations = len(model_specifications) * len(ml_models)

print("=" * 80)
print("MODEL EXECUTION")
print("=" * 80)
print(f"Total configuraciones: {len(model_specifications)}")
print(f"Modelos por configuración: {len(ml_models)}")
print(f"Total combinaciones: {total_combinations}")

print("\nModelos disponibles:")
print(f"   - LSTM: {'✅' if TENSORFLOW_AVAILABLE else '❌ (requiere tensorflow)'}")
print(f"   - ARIMA/SARIMA: {'✅' if STATSMODELS_AVAILABLE else '❌ (requiere statsmodels)'}")
print("=" * 80)

# Progress bar
pbar = tqdm(total=total_combinations, desc="Training models")

# Loop through all specifications
for spec_idx, spec in enumerate(model_specifications):

    num = spec['Num']
    aproximacion = spec['Aproximación']
    enfoque = spec['Enfoque']
    modelo = spec['Modelo']
    valores_a_usar = spec['Valores_a_usar']
    target = spec['Target']
    features = spec['Features']
    dataset_type = spec['Dataset']

    # Map variables to real column names
    target_col = var_mapping.get(target, target)
    feature_cols = map_variables(features)

    # Get data based on dataset type
    if dataset_type == 'global':
        df_to_use = global_df.copy()
        cluster_name = 'All'
    else:
        cluster_name = None  # Will be set in loop

    # Loop through ML models
    for ml_model in ml_models:

        # Skip if model not available
        if ml_model == 'LSTM' and not TENSORFLOW_AVAILABLE:
            pbar.update(1)
            continue
        if ml_model in ['ARIMA', 'SARIMA'] and not STATSMODELS_AVAILABLE:
            pbar.update(1)
            continue

        # Train model based on dataset type
        if dataset_type == 'global':

            # Train on global dataset
            try:
                if ml_model == 'LSTM':
                    metrics, trained_model = train_lstm_model(
                        df_to_use, feature_cols, target_col,
                        n_steps=7, epochs=50, verbose=0
                    )

                elif ml_model in ['ARIMA', 'SARIMA']:
                    if ml_model == 'SARIMA':
                        metrics, trained_model = train_arima_model(
                            df_to_use, feature_cols, target_col,
                            order=(1, 1, 1), verbose=0
                        )
                    else:  # ARIMA
                        metrics, trained_model = train_arima_model(
                            df_to_use, feature_cols, target_col,
                            order=(1, 1, 1), verbose=0
                        )

                # Store result
                result = {
                    'Num': num,
                    'Aproximación': aproximacion,
                    'Enfoque': enfoque,
                    'Modelo': modelo,
                    'Valores_a_usar': valores_a_usar,
                    'Target': f"{target}={target_col}",
                    'ML_Model': ml_model,
                    'Cluster': cluster_name,
                    **metrics
                }
                all_results.append(result)

            except Exception as e:
                # Store error result
                result = {
                    'Num': num,
                    'Aproximación': aproximacion,
                    'Enfoque': enfoque,
                    'Modelo': modelo,
                    'Valores_a_usar': valores_a_usar,
                    'Target': f"{target}={target_col}",
                    'ML_Model': ml_model,
                    'Cluster': cluster_name,
                    'RMSE': np.nan,
                    'MAE': np.nan,
                    'NSE': np.nan,
                    'KGE': np.nan,
                    'PBIAS_%': np.nan,
                    'MAPE_%': np.nan,
                    'Error': str(e)[:100]
                }
                all_results.append(result)

        else:  # dataset_type == 'clusters'

            # Train separate model for each cluster
            for cluster_name, cluster_df in cluster_data.items():
                try:
                    if ml_model == 'LSTM':
                        metrics, trained_model = train_lstm_model(
                            cluster_df, feature_cols, target_col,
                            n_steps=7, epochs=50, verbose=0
                        )

                    elif ml_model in ['ARIMA', 'SARIMA']:
                        if ml_model == 'SARIMA':
                            metrics, trained_model = train_arima_model(
                                cluster_df, feature_cols, target_col,
                                order=(1, 1, 1), verbose=0
                            )
                        else:  # ARIMA
                            metrics, trained_model = train_arima_model(
                                cluster_df, feature_cols, target_col,
                                order=(1, 1, 1), verbose=0
                            )

                    # Store result
                    result = {
                        'Num': num,
                        'Aproximación': aproximacion,
                        'Enfoque': enfoque,
                        'Modelo': modelo,
                        'Valores_a_usar': valores_a_usar,
                        'Target': f"{target}={target_col}",
                        'ML_Model': ml_model,
                        'Cluster': cluster_name,
                        **metrics
                    }
                    all_results.append(result)

                except Exception as e:
                    # Store error result
                    result = {
                        'Num': num,
                        'Aproximación': aproximacion,
                        'Enfoque': enfoque,
                        'Modelo': modelo,
                        'Valores_a_usar': valores_a_usar,
                        'Target': f"{target}={target_col}",
                        'ML_Model': ml_model,
                        'Cluster': cluster_name,
                        'RMSE': np.nan,
                        'MAE': np.nan,
                        'NSE': np.nan,
                        'KGE': np.nan,
                        'PBIAS_%': np.nan,
                        'MAPE_%': np.nan,
                        'Error': str(e)[:100]
                    }
                    all_results.append(result)

        pbar.update(1)

pbar.close()

print("\n" + "=" * 80)
print("EXECUTION COMPLETED")
print("=" * 80)
print(f"✅ Total models trained: {len(all_results)}")
print(f"   Successful: {sum(1 for r in all_results if not np.isnan(r.get('RMSE', np.nan)))}")
print(f"   Failed: {sum(1 for r in all_results if np.isnan(r.get('RMSE', np.nan)))}")


MODEL EXECUTION
Total configuraciones: 72
Modelos por configuración: 3
Total combinaciones: 216

Modelos disponibles:
   - LSTM: ✅
   - ARIMA/SARIMA: ✅


Training models: 100%|█████████████████████████████████████████████████████| 216/216 [1:03:06<00:00, 17.53s/it]


EXECUTION COMPLETED
✅ Total models trained: 504
   Successful: 504
   Failed: 0


## 13. Consolidación de Resultados

In [13]:
# Convert results to DataFrame
results_df = pd.DataFrame(all_results)

print("=" * 80)
print("RESULTS SUMMARY")
print("=" * 80)

print(f"\nDataFrame shape: {results_df.shape}")
print(f"\nColumns: {list(results_df.columns)}")
print(f"\nFirst 10 rows:")
display(results_df.head(10))

# Summary statistics
print("\n" + "=" * 80)
print("PERFORMANCE STATISTICS BY MODEL")
print("=" * 80)

for ml_model in results_df['ML_Model'].unique():
    model_results = results_df[results_df['ML_Model'] == ml_model]

    print(f"\n{ml_model}:")
    print(f"  Total: {len(model_results)}")
    print(f"  Valid: {model_results['RMSE'].notna().sum()}")
    print(f"  Failed: {model_results['RMSE'].isna().sum()}")

    if model_results['RMSE'].notna().any():
        print(f"\n  Metrics (mean ± std):")
        for metric in ['RMSE', 'MAE', 'NSE', 'KGE', 'PBIAS_%', 'MAPE_%']:
            if metric in model_results.columns:
                valid_vals = model_results[metric].dropna()
                if len(valid_vals) > 0:
                    mean_val = valid_vals.mean()
                    std_val = valid_vals.std()
                    print(f"    {metric:10s}: {mean_val:8.4f} ± {std_val:8.4f}")


RESULTS SUMMARY

DataFrame shape: (504, 14)

Columns: ['Num', 'Aproximación', 'Enfoque', 'Modelo', 'Valores_a_usar', 'Target', 'ML_Model', 'Cluster', 'RMSE', 'MAE', 'NSE', 'KGE', 'PBIAS_%', 'MAPE_%']

First 10 rows:


,Num,Aproximación,Enfoque,Modelo,Valores_a_usar,Target,ML_Model,Cluster,RMSE,MAE,NSE,KGE,PBIAS_%,MAPE_%
0,37,Series temporales,Todas las cuencas,Variables primarias,N/A,ymin=ymin,LSTM,All,0.180669,0.125695,0.139711,0.151084,9.108500,57.999522
1,37,Series temporales,Todas las cuencas,Variables primarias,N/A,ymin=ymin,ARIMA,All,0.063649,0.044809,0.451489,0.506239,11.582077,14.198093
2,37,Series temporales,Todas las cuencas,Variables primarias,N/A,ymin=ymin,SARIMA,All,0.063649,0.044809,0.451489,0.506239,11.582077,14.198093
3,38,Series temporales,Todas las cuencas,Variables primarias,N/A,yav=yav,LSTM,All,0.186434,0.131297,0.138958,0.140063,8.143464,58.875055
4,38,Series temporales,Todas las cuencas,Variables primarias,N/A,yav=yav,ARIMA,All,0.054889,0.039089,0.626289,0.608980,6.091533,12.395154
5,38,Series temporales,Todas las cuencas,Variables primarias,N/A,yav=yav,SARIMA,All,0.054889,0.039089,0.626289,0.608980,6.091533,12.395154
6,39,Series temporales,Todas las cuencas,Variables primarias,N/A,ymax=ymax,LSTM,All,0.193747,0.137198,0.147753,0.142869,6.728540,57.751401
7,39,Series temporales,Todas las cuencas,Variables primarias,N/A,ymax=ymax,ARIMA,All,0.054696,0.045230,0.663259,0.751245,-6.942591,15.423008
8,39,Series temporales,Todas las cuencas,Variables primarias,N/A,ymax=ymax,SARIMA,All,0.054696,0.045230,0.663259,0.751245,-6.942591,15.423008
9,40,Series temporales,Todas las cuencas,Variables primarias,N/A,Dif_y=Dif_y,LSTM,All,0.039725,0.024598,0.043793,-0.038259,-18.303546,112.130964



PERFORMANCE STATISTICS BY MODEL

LSTM:
  Total: 168
  Valid: 168
  Failed: 0

  Metrics (mean ± std):
    RMSE      :   0.7497 ±   0.8465
    MAE       :   0.5238 ±   0.6054
    NSE       :  -0.0472 ±   0.5521
    KGE       :   0.1958 ±   0.2363
    PBIAS_%   :  -7.3435 ±  35.0889
    MAPE_%    : 1681.4575 ± 4976.9059

ARIMA:
  Total: 168
  Valid: 168
  Failed: 0

  Metrics (mean ± std):
    RMSE      :   0.8421 ±   1.1496
    MAE       :   0.6886 ±   0.9746
    NSE       :  -0.3626 ±   1.0830
    KGE       :   0.2355 ±   0.3974
    PBIAS_%   : -27.1931 ±  39.6332
    MAPE_%    : 161.2923 ± 280.0849

SARIMA:
  Total: 168
  Valid: 168
  Failed: 0

  Metrics (mean ± std):
    RMSE      :   0.8421 ±   1.1496
    MAE       :   0.6886 ±   0.9746
    NSE       :  -0.3626 ±   1.0830
    KGE       :   0.2355 ±   0.3974
    PBIAS_%   : -27.1931 ±  39.6332
    MAPE_%    : 161.2923 ± 280.0849


## 14. Exportar Resultados a CSV

In [14]:
# Create output directory
output_dir = "../../data/model_results"
os.makedirs(output_dir, exist_ok=True)

# Generate timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Export to CSV
# csv_filename = f"notebook_6.1_time_series_models_{timestamp}.csv" # WITH OUTLIERS
csv_filename = f"notebook_6.1_time_series_models_{timestamp}-without-ouliers.csv" # WITHOUT OUTLIERS
csv_path = os.path.join(output_dir, csv_filename)

results_df.to_csv(csv_path, index=False)

print("=" * 80)
print("RESULTS EXPORTED")
print("=" * 80)

print("✅ Results saved to:")
print(f"   {csv_path}")

print(f"\n📊 File size: {os.path.getsize(csv_path) / 1024:.2f} KB")

print("\n📋 CSV structure:")
print(f"   Rows: {len(results_df)}")
print(f"   Columns: {len(results_df.columns)}")

print(f"\n🎯 Columns: {', '.join(results_df.columns)}")


RESULTS EXPORTED
✅ Results saved to:
   ../../data/model_results/notebook_6.1_time_series_models_20260216_231744-without-ouliers.csv

📊 File size: 104.85 KB

📋 CSV structure:
   Rows: 504
   Columns: 14

🎯 Columns: Num, Aproximación, Enfoque, Modelo, Valores_a_usar, Target, ML_Model, Cluster, RMSE, MAE, NSE, KGE, PBIAS_%, MAPE_%


## 15. Resumen Final y Conclusiones

In [15]:
print("=" * 80)
print("NOTEBOOK 6.1 - RESUMEN FINAL")
print("=" * 80)

print("\n📊 CONFIGURACIÓN:")
print(f"   - Configuraciones de series temporales: {len(model_specifications)} (filas 37-108)")
print(f"   - Modelos ML: {len(ml_models)} (LSTM, ARIMA, SARIMA)")
print(f"   - Total combinaciones: {len(model_specifications) * len(ml_models)}")

print("\n🎯 RESULTADOS:")
print(f"   - Modelos entrenados: {len(all_results)}")
print(f"   - Exitosos: {sum(1 for r in all_results if not np.isnan(r.get('RMSE', np.nan)))}")
print(f"   - Fallidos: {sum(1 for r in all_results if np.isnan(r.get('RMSE', np.nan)))}")

print("\n📁 OUTPUT:")
print(f"   - Archivo CSV: {csv_filename}")
print(f"   - Ubicación: {output_dir}")

print("\n✅ MÉTRICAS INCLUIDAS:")
print("   - RMSE (Root Mean Square Error)")
print("   - MAE (Mean Absolute Error)")
print("   - NSE (Nash-Sutcliffe Efficiency)")
print("   - KGE (Kling-Gupta Efficiency)")
print("   - PBIAS_% (Percent Bias)")
print("   - MAPE_% (Mean Absolute Percentage Error)")

print("\n🔬 MODELOS IMPLEMENTADOS:")
for ml_model in results_df['ML_Model'].unique():
    count = len(results_df[results_df['ML_Model'] == ml_model])
    success = results_df[results_df['ML_Model'] == ml_model]['RMSE'].notna().sum()
    print(f"   - {ml_model}: {success}/{count} exitosos")

print("\n" + "=" * 80)
print("Notebook 6.1 completado exitosamente ✅")
print("=" * 80)


NOTEBOOK 6.1 - RESUMEN FINAL

📊 CONFIGURACIÓN:
   - Configuraciones de series temporales: 72 (filas 37-108)
   - Modelos ML: 3 (LSTM, ARIMA, SARIMA)
   - Total combinaciones: 216

🎯 RESULTADOS:
   - Modelos entrenados: 504
   - Exitosos: 504
   - Fallidos: 0

📁 OUTPUT:
   - Archivo CSV: notebook_6.1_time_series_models_20260216_231744-without-ouliers.csv
   - Ubicación: ../../data/model_results

✅ MÉTRICAS INCLUIDAS:
   - RMSE (Root Mean Square Error)
   - MAE (Mean Absolute Error)
   - NSE (Nash-Sutcliffe Efficiency)
   - KGE (Kling-Gupta Efficiency)
   - PBIAS_% (Percent Bias)
   - MAPE_% (Mean Absolute Percentage Error)

🔬 MODELOS IMPLEMENTADOS:
   - LSTM: 168/168 exitosos
   - ARIMA: 168/168 exitosos
   - SARIMA: 168/168 exitosos

Notebook 6.1 completado exitosamente ✅
